# Whisper Large LoRA Run

Stage-driven notebook: configure, prepare data paths, load model, generate baseline predictions, evaluate, train LoRA, generate tuned predictions, evaluate, and write shared results.

## Configure Paths

This cell adds the repository root to `sys.path`, imports the run helpers, and builds the resolved `WhisperLargeRunConfig`. Replace the manifest paths with your real train, validation, and test CSVs before switching `smoke_mode` off.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "Runs").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Runs.whisper_large.utils import (
    WhisperLargeRunConfig,
    create_smoke_dataset_for_config,
    evaluate_saved_predictions,
    load_whisper_bundle,
    run_baseline_predictions_once,
    run_tuned_predictions,
    train_lora_with_early_stopping,
    write_stage_result,
)

config = WhisperLargeRunConfig(
    # Replace these three paths with the real ready-to-use dataset manifests.
    train_manifest=None,
    validation_manifest=None,
    test_manifest=None,
    output_dir=REPO_ROOT / "Runs" / "whisper_large" / "outputs",
    model_cache_dir=REPO_ROOT / "Runs" / "whisper_large" / "models" / "openai_whisper-large",
    model_name="openai/whisper-large",
    smoke_mode=True,
    patience=3,
    num_train_epochs=10,
    learning_rate=1e-4,
    optim="adamw_8bit",
).resolved()

config.to_dict()


[config] Output directory: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs
[config] Model cache directory: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\models\openai_whisper-large
[config] Model: openai/whisper-large
[config] Smoke mode: True


{'train_manifest': None,
 'validation_manifest': None,
 'test_manifest': None,
 'output_dir': 'C:\\Audio learning\\local_whisper\\Palestinian-ASR\\Runs\\whisper_large\\outputs',
 'model_cache_dir': 'C:\\Audio learning\\local_whisper\\Palestinian-ASR\\Runs\\whisper_large\\models\\openai_whisper-large',
 'model_name': 'openai/whisper-large',
 'model_key': 'whisper_large',
 'language': 'ar',
 'task': 'transcribe',
 'smoke_mode': True,
 'seed': 3407,
 'lora_r': 32,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'fc1', 'fc2'],
 'lora_bias': 'none',
 'gradient_checkpointing': 'unsloth',
 'optim': 'adamw_8bit',
 'learning_rate': 0.0001,
 'num_train_epochs': 10,
 'per_device_train_batch_size': 1,
 'per_device_eval_batch_size': 1,
 'gradient_accumulation_steps': 4,
 'patience': 3,
 'save_total_limit': 3,
 'resume_from_checkpoint': None,
 'generation_max_new_tokens': 128}

In [2]:
# In a real run, set smoke_mode=False above and provide real train/validation/test manifests.
if config.smoke_mode:
    config = create_smoke_dataset_for_config(config)

print("train_manifest:", config.train_manifest)
print("validation_manifest:", config.validation_manifest)
print("test_manifest:", config.test_manifest)


[stage] Creating smoke dataset and attaching manifests to config.
[smoke] Preparing smoke ASR dataset under: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\smoke_data
[smoke] train: audio=train_000.wav, manifest=C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\smoke_data\manifests\train.csv
[smoke] validation: audio=validation_000.wav, manifest=C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\smoke_data\manifests\validation.csv
[smoke] test: audio=test_000.wav, manifest=C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\smoke_data\manifests\test.csv
train_manifest: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\smoke_data\manifests\train.csv
validation_manifest: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\smoke_data\manifests\validation.csv
test_manifest: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outp

## Prepare Data or Smoke Dataset

This cell either creates the tiny smoke dataset used for wiring checks or confirms the real manifest paths are attached to the config. The printed paths are what the downstream stages consume.

In [3]:
# Baseline model load. In real mode this loads local cache or downloads openai/whisper-large.
baseline_bundle = load_whisper_bundle(config)
baseline_bundle


[model] Smoke mode enabled. Using deterministic smoke model; no Hugging Face download.


WhisperBundle(model=<Runs.whisper_large.utils.modeling.SmokeWhisperModel object at 0x000002437DA77D10>, processor=<Runs.whisper_large.utils.modeling.SmokeProcessor object at 0x000002437DAFCC20>, backend='smoke')

## Load Baseline Model

This loads the baseline Whisper bundle. In smoke mode it uses the deterministic local stand-in; in real mode it loads from the local cache or fetches `openai/whisper-large`.

In [4]:
# Generate baseline predictions once. Later notebook runs reuse the saved JSONL file.
baseline_predictions = run_baseline_predictions_once(config, bundle=baseline_bundle)
baseline_predictions


[baseline] Cached predictions not found. Generating baseline predictions now.
[data] Loading manifest: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\smoke_data\manifests\test.csv
[data] Loaded 1 rows from CSV.
[data] Resolved 1 records for split 'test'.
[predictions] Output file: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\predictions\baseline_test_predictions.jsonl
[predictions] Number of samples: 1


[predictions] Generating:   0%|          | 0/1 [00:00<?, ?sample/s]

[predictions] Generating: 100%|##########| 1/1 [00:00<00:00, 990.16sample/s]

[predictions] Finished writing predictions: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\predictions\baseline_test_predictions.jsonl


{'predictions_path': WindowsPath('C:/Audio learning/local_whisper/Palestinian-ASR/Runs/whisper_large/outputs/predictions/baseline_test_predictions.jsonl'),
 'cached': False}

## Generate Baseline Predictions

This runs inference for the baseline model on the test split and writes a cached JSONL file. Future notebook runs reuse that file instead of recomputing predictions.

In [5]:
# Evaluate the saved baseline predictions separately from generation.
baseline_metrics = evaluate_saved_predictions(
    config,
    predictions_path=baseline_predictions["predictions_path"],
    stage="baseline",
)
baseline_results_path = write_stage_result(
    config,
    stage="baseline",
    metrics=baseline_metrics,
    predictions_path=baseline_predictions["predictions_path"],
    metadata={"cached": baseline_predictions["cached"]},
)
baseline_metrics


[eval] Evaluating baseline predictions: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\predictions\baseline_test_predictions.jsonl
[eval] baseline metrics: WER=0.000000, CER=0.000000, RTF=3.3999967854470015e-06
[eval] Wrote metrics: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\metrics\baseline_metrics.json
[results] Writing baseline result into: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\results.json
[results] Updated shared results file: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\results.json


{'num_samples': 1,
 'cer': 0.0,
 'wer': 0.0,
 'rtf': 3.3999967854470015e-06,
 'total_audio_seconds': 1.0,
 'total_inference_seconds': 3.3999967854470015e-06}

## Evaluate Baseline Predictions

This evaluates the saved baseline JSONL, writes `baseline_metrics.json`, and updates the shared `results.json` record for the run.

In [6]:
# Train or resume LoRA. Real training evaluates only at the end of each epoch.
training_summary = train_lora_with_early_stopping(config)
training_summary


[training] Starting LoRA training stage.
[training] Smoke mode enabled. Creating placeholder checkpoint files.
[training] Wrote training summary: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\training_summary.json
[training] Smoke best checkpoint: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\checkpoints\whisper_large\best


{'backend': 'smoke',
 'best_checkpoint': 'C:\\Audio learning\\local_whisper\\Palestinian-ASR\\Runs\\whisper_large\\outputs\\checkpoints\\whisper_large\\best',
 'best_metric': 0.0,
 'completed_at': '2026-06-20T15:34:10.246628+00:00'}

## Train LoRA

This starts or resumes LoRA fine-tuning. Real runs evaluate at the end of each epoch and save checkpoints under `outputs/checkpoints/whisper_large/`.

In [7]:
# Generate predictions from the best checkpoint. Evaluation happens in the next cell from the saved JSONL file.
tuned_predictions = run_tuned_predictions(
    config,
    checkpoint_path=training_summary.get("best_checkpoint"),
)
tuned_predictions


[tuned] Generating tuned predictions with checkpoint: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\checkpoints\whisper_large\best
[data] Loading manifest: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\smoke_data\manifests\test.csv
[data] Loaded 1 rows from CSV.
[data] Resolved 1 records for split 'test'.
[predictions] Output file: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\predictions\tuned_test_predictions.jsonl
[predictions] Number of samples: 1
[model] Smoke mode enabled. Using deterministic smoke model; no Hugging Face download.


[predictions] Generating:   0%|          | 0/1 [00:00<?, ?sample/s]

[predictions] Generating: 100%|##########| 1/1 [00:00<00:00, 999.60sample/s]

[predictions] Finished writing predictions: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\predictions\tuned_test_predictions.jsonl


{'predictions_path': WindowsPath('C:/Audio learning/local_whisper/Palestinian-ASR/Runs/whisper_large/outputs/predictions/tuned_test_predictions.jsonl'),
 'checkpoint_path': 'C:\\Audio learning\\local_whisper\\Palestinian-ASR\\Runs\\whisper_large\\outputs\\checkpoints\\whisper_large\\best'}

## Generate Tuned Predictions

This loads the best checkpoint from training and writes tuned predictions to a separate JSONL file. The next cell evaluates that saved output.

In [8]:
# Evaluate tuned predictions from the saved JSONL file and write final shared model result.
tuned_metrics = evaluate_saved_predictions(
    config,
    predictions_path=tuned_predictions["predictions_path"],
    stage="tuned",
)
tuned_results_path = write_stage_result(
    config,
    stage="tuned",
    metrics=tuned_metrics,
    predictions_path=tuned_predictions["predictions_path"],
    metadata={"training": training_summary},
)
tuned_metrics


[eval] Evaluating tuned predictions: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\predictions\tuned_test_predictions.jsonl
[eval] tuned metrics: WER=0.000000, CER=0.000000, RTF=2.4999899324029684e-06
[eval] Wrote metrics: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\metrics\tuned_metrics.json
[results] Writing tuned result into: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\results.json
[results] Updated shared results file: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\results.json


{'num_samples': 1,
 'cer': 0.0,
 'wer': 0.0,
 'rtf': 2.4999899324029684e-06,
 'total_audio_seconds': 1.0,
 'total_inference_seconds': 2.4999899324029684e-06}

## Evaluate Tuned Predictions

This evaluates the tuned JSONL, writes `tuned_metrics.json`, and stores the tuned result in the shared run output.

## Summarize Results

This final cell prints the key outputs from the run so you can quickly verify the baseline metrics, training summary, tuned metrics, and final results path.

In [9]:
print("Baseline metrics:", baseline_metrics)
print("Training summary:", training_summary)
print("Tuned metrics:", tuned_metrics)
print("Shared results:", tuned_results_path)


Baseline metrics: {'num_samples': 1, 'cer': 0.0, 'wer': 0.0, 'rtf': 3.3999967854470015e-06, 'total_audio_seconds': 1.0, 'total_inference_seconds': 3.3999967854470015e-06}
Training summary: {'backend': 'smoke', 'best_checkpoint': 'C:\\Audio learning\\local_whisper\\Palestinian-ASR\\Runs\\whisper_large\\outputs\\checkpoints\\whisper_large\\best', 'best_metric': 0.0, 'completed_at': '2026-06-20T15:34:10.246628+00:00'}
Tuned metrics: {'num_samples': 1, 'cer': 0.0, 'wer': 0.0, 'rtf': 2.4999899324029684e-06, 'total_audio_seconds': 1.0, 'total_inference_seconds': 2.4999899324029684e-06}
Shared results: C:\Audio learning\local_whisper\Palestinian-ASR\Runs\whisper_large\outputs\results.json
